# 🍃 Exercise 1 · Your GPU workspace and a lab-trained baseline
**Operation CropGuard** · Kubeflow tutorial, part 1 of 5 · *Interactive computing*

## The mission

A farm-advisory service wants a phone app: **a farmer photographs a leaf and gets a diagnosis.**
The best public training data for this is **PlantVillage**: tens of thousands of leaf photos, all shot in a lab against plain backdrops.
Models trained on it look superb on lab photos. **Will they survive a real field?**

You will find out. In this notebook you will:

1. **Check your workspace.** This notebook is running inside a pod that Kubeflow scheduled for you, with a GPU and a data volume attached. You will see both.
2. **Find and sanity-check the data** that was pre-loaded onto your volume (no downloads in this tutorial).
3. **Fine-tune a ConvNeXt-Tiny** on lab photos using mixed precision on your GPU: the *baseline*.
4. **Test it honestly**: first on lab photos, then on **real smartphone photos of coffee leaves from a plantation** that the model never trained on.
5. **Try a first manual fix** (a data-augmentation recipe), re-train, and measure whether it helped.

By the end you will have a baseline, an honest picture of where it works and where it does not, and a feeling for how slow, fiddly and *noisy* manual tuning is.
That is exactly the problem the next exercises (Katib, distributed training, pipelines, serving) take off your hands.

## How long, and how to work

| Step | What | Compute time |
|---|---|---|
| 1 | Meet your workspace (GPU, CPU, memory, volumes) | seconds |
| 2 | Load and check the data | ~1 min |
| 3 | The model | seconds |
| 4 | Training machinery (read the code, run the cell) | seconds |
| 5 | **Baseline:** fine-tune on lab photos | a few min |
| 6 | **Field test** on real coffee photos | seconds |
| 7 | Why does it fail? (backdrop probe) | seconds |
| 8 | **Manual fix:** augmentation recipe, re-train, compare | a few min |
| 9 | Scorecard, save, reflect | seconds |

Python packages used: PyTorch and torchvision (CUDA build), numpy, pandas, matplotlib, scikit-learn, pillow. If an import fails, `pip install` it and re-run.

Run cells **top to bottom** with `Shift+Enter`. Callouts you will see:

> ✋ **Predict** — write a guess *before* you run the next cell. Guessing first is how you notice what surprised you.
> 🔧 **Try it** — optional experiment; the notebook works without it.
> 💡 **Why** — the reason behind a choice.
> ✅ **Checkpoint** — what you should see before moving on. If you don't, check *If something goes wrong* at the bottom.

Knobs live in **one cell** (the next one). Change them in a *second* run, not the first.

## The two datasets

| | PlantVillage (train) | RoCoLe (field test) |
|---|---|---|
| Photos | 7,629 leaf photos (about 200 per class, whole leaves) | 1,560 smartphone photos |
| Conditions | lab, plain backdrop, one leaf in the middle | a real coffee plantation in Ecuador: soil, weeds, shadows, harsh sun |
| Labels | 38 classes (14 crops; each class is a crop plus a condition, healthy or a disease) | healthy, coffee rust (4 levels), red spider mite |
| How we use it | train / validate / test, **split by leaf** | **never trained on**; used only to test |
| Licence | CC BY-SA 3.0 (per the Hugging Face card) | CC BY 4.0 |

**The catch:** coffee is not in PlantVillage, and neither are coffee's diseases. So the field test can only ask a **triage question: "healthy or sick?"**
We derive it from the 38-class model: `sick score = 1 − (sum of the probabilities of the 12 healthy classes)`.

## Setup: imports and the knobs

In [ ]:
import os, io, json, time, math, random, shutil, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch, torch.nn as nn, torch.nn.functional as F, torchvision
from PIL import Image, ImageFilter, ImageEnhance
from sklearn.metrics import roc_auc_score, balanced_accuracy_score

warnings.filterwarnings('ignore')
plt.rcParams.update({'figure.dpi': 110, 'axes.spines.top': False, 'axes.spines.right': False})
pd.set_option('display.max_colwidth', None)      # do not cut long row labels with '...'

# ---- Knobs. The defaults are what the instructors tested. -----------------------------------------------------------
SEED            = 42
EPOCHS          = 4                      # passes over the training photos
BATCH_SIZE      = 64                     # lower it (32, 16) if the GPU runs out of memory
LR              = 2e-4                   # peak learning rate (AdamW, warm-up then cosine decay)
WEIGHT_DECAY    = 0.05
LABEL_SMOOTHING = 0.1
RECIPE          = 'flip+rrc0.1+photo1'   # the augmentation recipe from step 8 (explained there)
NUM_WORKERS     = None                   # CPU processes that prepare batches (None = decide from the CPUs and shared memory this pod really has)
QUICK           = False                  # True = tiny smoke-test run (~1 min) to check that everything works

random.seed(SEED); np.random.seed(SEED); _ = torch.manual_seed(SEED)

---
## Step 1 · Meet your workspace

When you created this notebook server, Kubeflow asked Kubernetes for a **pod**: a container with a slice of a machine's CPUs, memory and **one GPU**, plus your **volumes** mounted as folders.
Everything you run here executes *there*, not on your laptop. Let's look at what you were given.

First, the GPU as the driver sees it:

In [ ]:
!nvidia-smi

Now the same GPU as **PyTorch** sees it. This is the check you will run at the start of every GPU session in your career.

In [ ]:
CUDA_OK = torch.cuda.is_available()
DEVICE = 'cuda' if CUDA_OK else 'cpu'
print(f'PyTorch {torch.__version__} | built for CUDA {torch.version.cuda} | cuDNN {torch.backends.cudnn.version()}')
if CUDA_OK:
    p = torch.cuda.get_device_properties(0)
    print(f'GPU 0: {p.name} | compute capability {p.major}.{p.minor} | {p.total_memory / 2**30:.1f} GiB | GPUs visible to this notebook: {torch.cuda.device_count()}')
    torch.backends.cudnn.benchmark = True        # let cuDNN pick the fastest convolution kernels for our fixed input size
else:
    print('⚠️  No GPU visible to PyTorch. This notebook switches to QUICK mode so it still runs, but the numbers will be meaningless.')
    print('    On the tutorial cluster this means the notebook server was created without a GPU: ask an instructor.')
    QUICK = True

**Why does the GPU matter?** A quick race: the same matrix multiplication (the core operation of neural networks) on the CPU and on the GPU.

In [ ]:
def matmul_tflops(device, dtype, n=2048, reps=20):
    a = torch.randn(n, n, device=device, dtype=dtype); b = torch.randn(n, n, device=device, dtype=dtype)
    for _ in range(3): a @ b                                   # warm-up
    if device == 'cuda': torch.cuda.synchronize()             # GPU calls are asynchronous: wait for them before reading the clock
    t0 = time.time()
    for _ in range(reps): a @ b
    if device == 'cuda': torch.cuda.synchronize()
    return 2 * n**3 * reps / (time.time() - t0) / 1e12

cpu = matmul_tflops('cpu', torch.float32)
print(f'CPU  fp32 : {cpu:7.2f} TFLOP/s')
if CUDA_OK:
    g32, g16 = matmul_tflops('cuda', torch.float32), matmul_tflops('cuda', torch.float16)
    print(f'GPU  fp32 : {g32:7.2f} TFLOP/s   ({g32 / cpu:,.0f}x the CPU)')
    print(f'GPU  fp16 : {g16:7.2f} TFLOP/s   ({g16 / cpu:,.0f}x the CPU)  <- tensor cores; this is why we train in mixed precision')

> 💡 **Why mixed precision (fp16)?** GPUs have dedicated tensor cores that run 16-bit maths much faster, and 16-bit numbers also halve memory traffic.
> PyTorch's *autocast* picks fp16 where it is safe and keeps fp32 where it is not; a *GradScaler* stops tiny fp16 gradients from rounding to zero.
> The NVIDIA **T4** you were given supports fp16 but **not** bf16, so we use fp16 everywhere.

Next, the rest of the pod: CPU cores, memory limit, and the volumes mounted into it.

In [ ]:
def cgroup(path):
    try: return Path(path).read_text().strip()
    except Exception: return 'n/a'

def available_cpus():
    """CPU cores this pod may really use: the cgroup quota if one is set (Kubernetes CPU limit), otherwise the cores the OS shows."""
    try:
        quota, period = cgroup('/sys/fs/cgroup/cpu.max').split()
        if quota != 'max': return max(1, math.ceil(int(quota) / int(period)))
    except Exception: pass
    return len(os.sched_getaffinity(0))

CPUS = available_cpus()
SHM_MB = shutil.disk_usage('/dev/shm').total / 2**20 if Path('/dev/shm').exists() else None     # DataLoader workers hand batches to the trainer through shared memory
PREFETCH = 4                                                                                      # batches each worker keeps ready
if NUM_WORKERS is None: NUM_WORKERS = min(8, max(1, CPUS - 1))                                    # leave one core for the notebook itself
if SHM_MB is not None and SHM_MB < 512:
    NUM_WORKERS, PREFETCH = min(NUM_WORKERS, 2), 2
    print(f'⚠️  shared memory (/dev/shm) is only {SHM_MB:,.0f} MB: using {NUM_WORKERS} data-loader workers and a short prefetch so batches fit. (Kubeflow: tick "enable shared memory" when creating a notebook server.)')
print('CPU cores this notebook may use :', CPUS, '| cgroup cpu.max:', cgroup('/sys/fs/cgroup/cpu.max'))
print('memory limit (cgroup)           :', cgroup('/sys/fs/cgroup/memory.max'), '(bytes; "max" = no limit set)')
print('shared memory (/dev/shm)        :', f'{SHM_MB:,.0f} MB' if SHM_MB is not None else 'n/a')
print('data-loader workers we will use :', NUM_WORKERS)
!df -hT -x tmpfs -x devtmpfs -x squashfs -x overlay 2>/dev/null

> ✅ **Checkpoint 1.** `CUDA_OK` is `True`, the GPU name is printed, and the fp16 line is several times faster than fp32.
> Your volumes appear in the `df` table (one is your home/workspace, one holds the pre-loaded data).
>
> 🔧 **Try it.** Open a **terminal** in JupyterLab (File → New → Terminal) and run `watch -n 1 nvidia-smi`. Keep it visible: you will see the GPU wake up when training starts in step 5.

---
## Step 2 · Find and check the data

The dataset was copied onto a **PersistentVolumeClaim (PVC)** that is mounted into your pod, so nobody downloads gigabytes of photos over the venue Wi-Fi.
The cell below looks for it in the usual places. If your instructor mounted it somewhere else, set the environment variable `CROPGUARD_DATA` or add the path to `CANDIDATES`.

In [ ]:
CANDIDATES = [os.environ.get('CROPGUARD_DATA'), '/data/cropguard', '/home/jovyan/data/cropguard', '../data/cropguard', 'data/cropguard']
DATA = next((Path(c) for c in CANDIDATES if c and (Path(c) / 'MANIFEST.json').exists()), None)
if DATA is None:
    raise FileNotFoundError('Could not find the CropGuard data volume (a folder containing MANIFEST.json). Looked in: '
                            + ', '.join(str(c) for c in CANDIDATES if c) + '. Ask an instructor where the PVC is mounted, then set CROPGUARD_DATA.')
MANIFEST = json.loads((DATA / 'MANIFEST.json').read_text())
print('data volume:', DATA.resolve(), f"| {MANIFEST['total_gb']} GB | built {MANIFEST['built']}")
print(pd.Series({k: f'{v / 1e6:,.0f} MB' for k, v in MANIFEST['bytes'].items()}).to_string())

Everything the notebook needs is there: two image arrays, two small tables describing them, and the pretrained model weights.
Reading the arrays into memory also tells you how fast your volume is (a second read is much faster, because the operating system keeps the file in memory).

In [ ]:
t0 = time.time()
pv = pd.read_csv(DATA / 'plantvillage/meta.csv', index_col='idx'); ro = pd.read_csv(DATA / 'rocole/meta.csv', index_col='idx')
pv_img = np.load(DATA / 'plantvillage/images.npy'); ro_img = np.load(DATA / 'rocole/images.npy')
mb = (pv_img.nbytes + ro_img.nbytes) / 1e6; dt = time.time() - t0
print(f'read {mb:,.0f} MB from the volume in {dt:.1f}s ({mb / dt:,.0f} MB/s)')

CLASSES = MANIFEST['classes']; N_CLASSES = len(CLASSES); HEALTHY = np.array(MANIFEST['healthy_class_ids'])
y = pv['cls'].map({c: i for i, c in enumerate(CLASSES)}).values                           # class name -> integer label
TRAIN, VAL, TEST = [np.where(pv['split'] == s)[0] for s in ('train', 'val', 'test')]
ro_sick = ro['sick'].values.astype(int); plants = ro['plant'].values
DEV, LOCKED = [(ro['half'] == h).values for h in ('dev', 'locked')]
if QUICK: TRAIN = np.random.RandomState(SEED).permutation(TRAIN)[:512]; EPOCHS = 1      # smoke-test mode

print(f'\nPlantVillage: {len(pv):,} photos | {N_CLASSES} classes ({len(HEALTHY)} healthy) | train {len(TRAIN):,}  val {len(VAL):,}  test {len(TEST):,}')
print(f'RoCoLe      : {len(ro):,} photos | {len(np.unique(plants))} plants | healthy {int((1 - ro_sick).sum())}, sick {int(ro_sick.sum())} | dev half {DEV.sum()}  locked half {LOCKED.sum()}')

Two **splitting rules** are baked into those tables. They are the most important thing to understand about this dataset:

* **PlantVillage is split by *leaf*.** The same physical leaf was photographed several times from slightly different angles. If sibling photos land on both sides of a train/test split,
  the test is partly a memory test, and accuracy is inflated. (The split that ships with the dataset has this problem: about 8% of its test photos have a sibling in train.) Our `split` column keeps every leaf on one side.
* **RoCoLe is split by *plant* into a `dev` half and a `locked` half.** There are four photos per coffee plant, so the same logic applies. The **`dev` half is for making choices** (a recipe, a setting, a cut-off),
  and the **`locked` half is the final exam**: nothing is ever chosen using it. Today you report on both, and you will tune one cut-off on `dev` and judge it on `locked`.

The cell below proves the splits are clean.

In [ ]:
assert pv_img.shape[0] == len(pv) and ro_img.shape[0] == len(ro) and pv_img.shape[1:] == (224, 224, 3)
leaf_overlap = len(set(pv['gid'].values[TRAIN]) & set(pv['gid'].values[TEST])) + len(set(pv['gid'].values[TRAIN]) & set(pv['gid'].values[VAL]))
plant_overlap = len(set(plants[DEV]) & set(plants[LOCKED]))
assert leaf_overlap == 0 and plant_overlap == 0, 'splits leak!'
print(f'✅ leaves shared between train and val/test: {leaf_overlap} | coffee plants shared between dev and locked: {plant_overlap}')
counts = pv.groupby('cls').size(); print(f'photos per class: min {counts.min()}, median {int(counts.median())}, max {counts.max()} (capped at 200 per class so no class dominates)')

Now **look at the data**. Always do this before training anything.

In [ ]:
def show(imgs, titles, ncols=8, cell=1.9, fs=7):
    nrows = math.ceil(len(imgs) / ncols); fig, ax = plt.subplots(nrows, ncols, figsize=(cell * ncols, cell * nrows * 1.08), squeeze=False)
    for a in ax.ravel(): a.axis('off')
    for a, im, t in zip(ax.ravel(), imgs, titles): a.imshow(im); a.set_title(t, fontsize=fs)
    plt.tight_layout(); plt.show()

rs = np.random.RandomState(1)
lab = rs.choice(TRAIN, 8, replace=False)
heal, sick_ = rs.choice(np.where(~ro_sick.astype(bool))[0], 4, replace=False), rs.choice(np.where(ro_sick.astype(bool))[0], 4, replace=False)
show(list(pv_img[lab]) + list(ro_img[heal]) + list(ro_img[sick_]),
     [pv['cls'].iloc[i].replace('___', '\n').replace('_', ' ')[:30] for i in lab] + ['RoCoLe: ' + ro['cls'].iloc[i] for i in list(heal) + list(sick_)])

**Top row: PlantVillage. Bottom rows: RoCoLe.** Notice what changes between the worlds: the lab backdrop (grey/black, perfectly even) versus soil, weeds and shadows;
one centred, whole leaf versus partial, tilted, sun-bleached leaves; and a high-end lab camera versus a phone.

> ✅ **Checkpoint 2.** You see 7,629 + 1,560 photos, zero leaves shared between splits, and a clear visual gap between the rows. If the assert fails, stop and ask an instructor.

---
## Step 3 · The model: ConvNeXt-Tiny

We fine-tune **ConvNeXt-Tiny** (about 28 M parameters), a modern convolutional network pretrained on ImageNet-1k (1.3 M everyday photos). *Pretrained* means it already knows edges, textures and shapes;
we only have to teach it what 38 leaf conditions look like.

> 💡 **Why this backbone?** In our research phase we compared five pretrained networks on the same real-world test. **The choice of backbone mattered more than any augmentation trick:**
> older ResNets were weak at spotting sick coffee leaves, while ConvNeXt and DINOv2 features were clearly better. ConvNeXt-Tiny is a good fit for a 16 GB T4: strong, and small enough to train in minutes.

The weights come from the data volume (no internet needed). We swap the final layer, which was a 1,000-way ImageNet classifier, for a fresh 38-way one.

In [ ]:
WEIGHTS = DATA / 'weights/convnext_tiny_imagenet1k_v1.pth'

def make_model(num_classes=N_CLASSES):
    m = torchvision.models.convnext_tiny(weights=None)
    m.load_state_dict(torch.load(WEIGHTS, map_location='cpu', weights_only=True))     # ImageNet-pretrained weights
    m.classifier[2] = nn.Linear(768, num_classes)                                       # new head: 768 features -> 38 classes
    return m

m = make_model()
print(f'ConvNeXt-Tiny: {sum(p.numel() for p in m.parameters()) / 1e6:.1f} M parameters | new head: {m.classifier[2]}')
print('stages:', [type(c).__name__ + f'({len(c)})' if hasattr(c, '__len__') else type(c).__name__ for c in m.features])
del m

---
## Step 4 · The training machinery

Read this cell once; you will reuse these ideas in every exercise. Four pieces:

1. **`LeafDataset`** hands out `(photo, label)` pairs. It can also apply an *augmentation function* to a photo (empty for the baseline).
2. A **`DataLoader`** with `NUM_WORKERS` CPU processes prepares batches in parallel *while the GPU trains on the previous one*, so the expensive GPU never waits. Photos travel as small `uint8` arrays;
   the conversion to normalised floats happens **on the GPU** (`to_gpu`).
3. **`fit()`** is the training loop: forward pass in fp16 (`autocast`), backward pass with a `GradScaler`, AdamW with a warm-up-then-cosine learning-rate schedule, gradient clipping.
   After each epoch it scores the **validation** photos so you can watch it learn.
4. **`predict_proba()`** turns photos into class probabilities, in batches, without gradients.

In [ ]:
MEAN = torch.tensor([0.485, 0.456, 0.406], device=DEVICE).view(1, 3, 1, 1) * 255          # ImageNet statistics (the pretrained model expects them)
STD = torch.tensor([0.229, 0.224, 0.225], device=DEVICE).view(1, 3, 1, 1) * 255

def to_gpu(batch_u8):
    """uint8 (B,H,W,3) tensor on the CPU -> normalised float (B,3,H,W) on the device, channels-last (fastest layout for convolutions on tensor cores)."""
    x = batch_u8.to(DEVICE, non_blocking=True).permute(0, 3, 1, 2).float()
    return ((x - MEAN) / STD).contiguous(memory_format=torch.channels_last)

class LeafDataset(torch.utils.data.Dataset):
    """Serves (uint8 photo, label). With `augment`, every photo is served twice per epoch: once untouched, once freshly augmented."""
    def __init__(self, images, labels, idx, augment=None):
        self.images, self.labels, self.augment, self.epoch = images, labels, augment, 0
        self.items = [(int(i), False) for i in idx] + ([(int(i), True) for i in idx] if augment else [])
    def __len__(self): return len(self.items)
    def __getitem__(self, k):
        i, aug = self.items[k]; img = self.images[i]
        if aug: img = self.augment(img, np.random.default_rng([SEED, self.epoch, i]))   # a new random draw each epoch, reproducible from (seed, epoch, photo)
        return torch.from_numpy(np.ascontiguousarray(img)), int(self.labels[i])

@torch.inference_mode()
def predict_proba(model, images, bs=128):
    model.eval(); out = []
    for i in range(0, len(images), bs):
        with torch.autocast(DEVICE, dtype=torch.float16, enabled=CUDA_OK):
            out.append(F.softmax(model(to_gpu(torch.from_numpy(np.ascontiguousarray(images[i:i + bs])))).float(), 1).cpu())
    return torch.cat(out).numpy()

def fit(augment=None, epochs=EPOCHS, lr=LR, batch=BATCH_SIZE, seed=SEED):
    torch.manual_seed(seed)
    model = make_model().to(DEVICE).to(memory_format=torch.channels_last)
    ds = LeafDataset(pv_img, y, TRAIN, augment)
    dl = torch.utils.data.DataLoader(ds, batch_size=batch, shuffle=True, drop_last=True, num_workers=NUM_WORKERS, pin_memory=CUDA_OK,
                                     prefetch_factor=PREFETCH if NUM_WORKERS else None, generator=torch.Generator().manual_seed(seed))
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=WEIGHT_DECAY)
    total, warm = epochs * len(dl), max(10, int(0.1 * epochs * len(dl)))
    sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: min(1, (s + 1) / warm) * 0.5 * (1 + math.cos(math.pi * min(1, s / total))))
    scaler = torch.amp.GradScaler(enabled=CUDA_OK)
    if CUDA_OK: torch.cuda.reset_peak_memory_stats()
    hist, t_start = [], time.time()
    for ep in range(epochs):
        ds.epoch = ep; model.train(); t0 = time.time(); loss_sum = torch.zeros((), device=DEVICE); seen = 0
        for xb, yb in dl:
            x, t = to_gpu(xb), yb.to(DEVICE, non_blocking=True)
            with torch.autocast(DEVICE, dtype=torch.float16, enabled=CUDA_OK):
                loss = F.cross_entropy(model(x), t, label_smoothing=LABEL_SMOOTHING)
            opt.zero_grad(set_to_none=True); scaler.scale(loss).backward()
            scaler.unscale_(opt); nn.utils.clip_grad_norm_(model.parameters(), 1.0)         # clip after unscaling so the threshold is in real units
            scaler.step(opt); scaler.update(); sched.step()
            loss_sum += loss.detach() * len(t); seen += len(t)
        if CUDA_OK: torch.cuda.synchronize()
        secs = time.time() - t0; val_acc = float((predict_proba(model, pv_img[VAL]).argmax(1) == y[VAL]).mean())
        hist.append({'epoch': ep + 1, 'train_loss': float(loss_sum) / seen, 'val_acc': val_acc, 'seconds': secs, 'img_per_s': seen / secs})
        print(f'epoch {ep + 1}/{epochs}  train loss {hist[-1]["train_loss"]:.3f}  val acc {val_acc:.3f}  {secs:5.1f}s  ({seen / secs:,.0f} photos/s)', flush=True)
    info = {'epochs': epochs, 'photos_per_epoch': len(ds), 'train_seconds': time.time() - t_start, 'peak_gpu_gb': torch.cuda.max_memory_allocated() / 2**30 if CUDA_OK else 0.0}
    return model.eval(), pd.DataFrame(hist), info

print('training machinery defined')

**A speed check before the real run.** We time a few training steps on random data to see how fast *this* GPU is, and project how long the baseline will take.
(If the projection looks too long on your hardware, lower `EPOCHS` in the knobs cell and re-run from there.)

In [ ]:
def gpu_speed_check(steps=12, batch=BATCH_SIZE):
    m = make_model().to(DEVICE).to(memory_format=torch.channels_last); opt = torch.optim.AdamW(m.parameters(), lr=1e-5); scaler = torch.amp.GradScaler(enabled=CUDA_OK)
    x = torch.randn(batch, 3, 224, 224, device=DEVICE).contiguous(memory_format=torch.channels_last); t = torch.randint(0, N_CLASSES, (batch,), device=DEVICE)
    for s in range(steps + 3):
        if s == 3:
            if CUDA_OK: torch.cuda.synchronize()
            t0 = time.time()
        with torch.autocast(DEVICE, dtype=torch.float16, enabled=CUDA_OK): loss = F.cross_entropy(m(x), t)
        opt.zero_grad(); scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
    if CUDA_OK: torch.cuda.synchronize()
    del m, opt; torch.cuda.empty_cache() if CUDA_OK else None
    return steps * batch / (time.time() - t0)

GPU_IMG_S = gpu_speed_check()
print(f'{DEVICE.upper()} training speed on random data: ~{GPU_IMG_S:,.0f} photos/s')
print(f'projected baseline run : {EPOCHS} epochs x {len(TRAIN):,} photos  = ~{EPOCHS * len(TRAIN) / GPU_IMG_S / 60:.1f} min  (plus a validation pass per epoch)')

---
## Step 5 · The baseline: train on lab photos only

This is the model a team would build first: take the lab dataset, fine-tune, check accuracy on held-out lab photos, ship.

> ✋ **Predict.** Before you run the next cell, double-click here and write your guesses:
>
> * Accuracy on held-out **lab** photos (38 classes; chance is 2.6%): ______ %
> * Ability to tell **healthy from sick coffee leaves** in the field, as an AUC (0.5 = coin flip, 1.0 = perfect): ______
>
> *AUC* is the probability that a randomly chosen sick leaf gets a higher sick-score than a randomly chosen healthy one.

In [ ]:
t_run = time.time()
baseline, hist_base, info_base = fit(augment=None)
print(f'\nbaseline trained in {info_base["train_seconds"]:.0f}s | peak GPU memory {info_base["peak_gpu_gb"]:.1f} GiB')

While that trained, `val acc` is accuracy on **validation** photos (held-out leaves from the lab dataset). Now the real **test** with leaves it never saw, still from the lab:

In [ ]:
def triage_score(P): return 1 - P[:, HEALTHY].sum(1)         # P(sick) = 1 - sum of the 12 healthy classes' probabilities

def best_cutoff(s, mask):
    """The cut-off on the sick score that maximises balanced accuracy on the photos selected by `mask`."""
    cuts = np.linspace(.02, .98, 97)
    return float(cuts[np.argmax([balanced_accuracy_score(ro_sick[mask], s[mask] >= c) for c in cuts])])

CUT_KEY = 'cut-off tuned on the dev half'

def evaluate(model):
    P_te, P_ro = predict_proba(model, pv_img[TEST]), predict_proba(model, ro_img)
    s = triage_score(P_ro); halves = (('dev', DEV), ('LOCKED', LOCKED))
    r = {'lab: 38-class accuracy (PlantVillage test)': float((P_te.argmax(1) == y[TEST]).mean())}
    for name, mask in halves: r[f'field {name}: sick-vs-healthy AUC (ranking)'] = roc_auc_score(ro_sick[mask], s[mask])
    for name, mask in halves: r[f'field {name}: balanced accuracy at the default 0.5 cut-off'] = balanced_accuracy_score(ro_sick[mask], s[mask] >= .5)
    for name, mask in halves: r[f'field {name}: share called sick at 0.5 (truth {ro_sick[mask].mean():.0%})'] = float((s[mask] >= .5).mean())
    cut = best_cutoff(s, DEV)                                   # chosen on the dev half only ...
    r[CUT_KEY] = cut
    r['field LOCKED: balanced accuracy at that cut-off'] = balanced_accuracy_score(ro_sick[LOCKED], s[LOCKED] >= cut)     # ... and judged on the locked half
    return r, s

res_base, score_base = evaluate(baseline)
print(f"lab test accuracy: {res_base['lab: 38-class accuracy (PlantVillage test)']:.1%}")

## Step 6 · The field test

Now the question the farm-advisory service cares about: the same model, on **real phone photos of coffee leaves** it has never seen, from a plantation in another continent.
We ask only for the triage answer, *healthy or sick?* (see the table at the top for why).

The table reports the same photos through three lenses. They answer different questions, and a model can do well on one and badly on another:

| Lens | The question it answers |
|---|---|
| **AUC** | *Ranking.* Does a sick leaf get a higher sick-score than a healthy one? 0.5 = coin flip, 1.0 = perfect. It ignores where you draw the line. |
| **Balanced accuracy at 0.5** | *The default decision.* If the app says "sick" whenever the score is at least 0.5, how often is it right, averaged over healthy and sick leaves? 0.5 = no better than always giving the same answer. |
| **Cut-off tuned on dev** | *A fairer decision.* Choose the best cut-off using only the **dev** half, then judge it on the **locked** half. This is what the two halves are for. |

In [ ]:
pd.Series(res_base, name='baseline').to_frame().round(3)

In [ ]:
def score_hist(scores_by_model, cuts=None, mask=LOCKED, title='LOCKED half'):
    fig, ax = plt.subplots(1, len(scores_by_model), figsize=(5.4 * len(scores_by_model), 3.4), squeeze=False)
    for a, (name, s) in zip(ax[0], scores_by_model.items()):
        a.hist(s[mask & (ro_sick == 0)], bins=25, range=(0, 1), alpha=.65, label='healthy leaves', color='#2a9d5c'); a.hist(s[mask & (ro_sick == 1)], bins=25, range=(0, 1), alpha=.65, label='sick leaves', color='#d9625a')
        a.axvline(.5, color='k', lw=1, ls=':', label='default cut-off 0.5')
        if cuts: a.axvline(cuts[name], color='k', lw=1.2, ls='--', label=f'cut-off tuned on dev ({cuts[name]:.2f})')
        a.set_title(f'{name}: sick score, {title}', fontsize=10); a.set_xlabel('sick score (0 = confident healthy, 1 = confident sick)'); a.legend(frameon=False, fontsize=8)
    plt.tight_layout(); plt.show()
score_hist({'baseline': score_base}, cuts={'baseline': res_base[CUT_KEY]})

Each bar counts coffee leaves with a given sick-score. A useful model puts the red (sick) bars to the right of the green (healthy) ones; a *usable* one also has the cut-off line between them. Where are *your* bars relative to the two lines?

**Look at the mistakes.** Numbers hide stories; photos reveal them. Top row: *healthy* coffee leaves the baseline is most sure are sick. Bottom row: *sick* leaves it is most sure are healthy.

In [ ]:
def error_gallery(scores, n=8):
    fa = np.where(LOCKED & (ro_sick == 0))[0]; fa = fa[np.argsort(-scores[fa])][:n]            # healthy, highest sick score
    ms = np.where(LOCKED & (ro_sick == 1))[0]; ms = ms[np.argsort(scores[ms])][:n]               # sick, lowest sick score
    show(list(ro_img[fa]) + list(ro_img[ms]), [f'healthy, score {scores[i]:.2f}' for i in fa] + [f'{ro["cls"].iloc[i]}, score {scores[i]:.2f}' for i in ms], ncols=n)
error_gallery(score_base)

### Reading the baseline honestly

On the instructors' runs (several seeds, on an NVIDIA GB10 GPU; yours will differ somewhat) the baseline looked like this:

* **On lab photos it is excellent**: about 97–98% of the 38-way answers are right. If that were the only test, you would ship it.
* **On field photos its *ranking* is decent, not great** (AUC roughly 0.8–0.9): most of the time a sick leaf scores higher than a healthy one. For scale, in our research phase *frozen* ConvNeXt features
  scored about 0.71 on the dev half and frozen ResNet-18 features about 0.57 with the same test, so fine-tuning the whole network helped a lot.
* **But its default decision is close to useless.** In the histogram, nearly *every* coffee leaf, healthy or sick, scores above 0.5. At the default cut-off the app says "sick" for almost everything
  (balanced accuracy near 0.5: a coin flip). A plausible reason: coffee is not among the 14 crops it learned, so an unfamiliar leaf spreads its probability over the disease classes and no healthy class ever "wins".
* **A cut-off tuned on the dev half recovers much of it**: the model knows more than its default decision shows. But that cut-off was tuned on *this* plantation, so it fixes *this* field, not fields in general.
* **Lab accuracy and field AUC are different numbers on different tests.** Do not subtract one from the other. The lesson is that *the lab test could not have told you any of this.*

> ✅ **Checkpoint 3.** Lab accuracy in the high 90s; field AUC clearly below it; and the app calls nearly every coffee leaf "sick" at the default cut-off. Compare with your prediction.

---
## Step 7 · Why does it fail? Reading leaves or backdrops?

One reason lab models stumble outside the lab: in PlantVillage most classes were photographed in one *session* with one backdrop, so **the backdrop (and lighting) leaks the label**.
A model can score well by reading the background and barely looking at the leaf.

A direct test on the model you just trained, using the leaf masks that ship with the dataset. We feed it three versions of each test photo:
the **whole photo**, the **backdrop only** (leaf painted over with the average backdrop colour), and the **leaf only** (backdrop blacked out).

In [ ]:
test_masks = np.unpackbits(np.load(DATA / 'plantvillage/masks.npy')[TEST], axis=-1).astype(bool)

def backdrop_only(img, m):
    grown = np.asarray(Image.fromarray((m * 255).astype('uint8')).filter(ImageFilter.MaxFilter(7))) > 0      # grow the mask a little to cover the leaf edge
    out = img.copy(); out[grown] = img[~m].mean(0) if (~m).any() else img.mean((0, 1)); return out

X_bg = np.stack([backdrop_only(a, m) for a, m in zip(pv_img[TEST], test_masks)])
X_leaf = np.stack([a * m[..., None] for a, m in zip(pv_img[TEST], test_masks)])
show(list(pv_img[TEST][:5]) + list(X_bg[:5]) + list(X_leaf[:5]), ['whole photo'] * 5 + ['backdrop only'] * 5 + ['leaf only'] * 5, ncols=5, cell=2.2)

def probe(model):
    return {'whole photo': float((predict_proba(model, pv_img[TEST]).argmax(1) == y[TEST]).mean()),
            'backdrop only (no leaf!)': float((predict_proba(model, X_bg).argmax(1) == y[TEST]).mean()),
            'leaf only (no backdrop)': float((predict_proba(model, X_leaf).argmax(1) == y[TEST]).mean())}
probe_base = probe(baseline)
print(f'38-class accuracy on lab test photos (chance = {1 / N_CLASSES:.1%}):'); print(pd.Series(probe_base, name='baseline').map('{:.1%}'.format).to_string())

With **no leaf in the picture at all**, a model that was supposed to diagnose leaves still gets a surprising share right: on the instructors' runs about 14–27% against 2.6% for chance, i.e. five to ten times better than guessing.
That is the shortcut: in PlantVillage the backdrop and lighting partly give the class away. With the backdrop removed and only the leaf left, accuracy stays high (about 80–93%), so the leaf carries most of the real signal,
but the model also leans on the context. (Painting over a leaf makes the picture strange for the model, so treat the exact figures as a rough indicator, not a precise measurement of "how much it cheats".)
Real fields have no lab backdrop, so a model that leans on one has less to lean on there.

---
## Step 8 · A first manual fix: make lab photos look more like the field

If the problem is that lab photos are too clean, **augment** them: show the model each training photo again with random changes that mimic field conditions. We used our research phase to try ten recipes
on the real coffee photos (on the development half only). The recipe that came out on top is called `flip+rrc0.1+photo1`, built from three simple ingredients, applied in order:

| Ingredient | What it does | Why it should help |
|---|---|---|
| `flip` | random 90° rotation and mirror | phone photos have no fixed "up" |
| `rrc0.1` | **random resized crop**: zoom into a random region covering at least 10% of the photo, resize back | field photos are close-ups and partial leaves, not whole leaves centred in frame |
| `photo1` | **photometric jitter** at full strength: brightness, contrast, colour, hue, exposure, colour temperature, occasional blur | harsh sun, shade and a cheap lens |

**What the research found** (frozen ResNet-18, 195 development plants, so treat it as a hint rather than a promise):
adding this recipe moved the field AUC from ~0.58 to ~0.75. Fancier recipes that pasted leaves onto synthetic soil and grass backgrounds did *not* do better.
**But**, a bigger finding qualified it: the gain depended on the backbone, and it was largest for the weakest one. Whether it still helps a *fine-tuned ConvNeXt* is exactly what you are about to measure.

Here is the whole recipe in code. It is ~25 lines.

In [ ]:
def _flip_rot90(a, rng):
    a = np.rot90(a, int(rng.integers(0, 4)))
    return a[:, ::-1] if rng.random() < .5 else a

def random_resized_crop(img, rng, min_area=.25, ratio=(3 / 4, 4 / 3)):
    H = W = img.shape[0]; area = H * W * rng.uniform(min_area, 1.0); ar = np.exp(rng.uniform(np.log(ratio[0]), np.log(ratio[1])))
    w, h = min(W, int(round(np.sqrt(area * ar)))), min(H, int(round(np.sqrt(area / ar))))
    x0, y0 = int(rng.integers(0, W - w + 1)), int(rng.integers(0, H - h + 1))
    return np.asarray(Image.fromarray(np.ascontiguousarray(img)).crop((x0, y0, x0 + w, y0 + h)).resize((W, H), Image.BICUBIC))

def photometric(img, rng, s=1.0):
    im = Image.fromarray(np.ascontiguousarray(img))
    im = ImageEnhance.Brightness(im).enhance(1 + rng.uniform(-.45, .45) * s); im = ImageEnhance.Contrast(im).enhance(1 + rng.uniform(-.4, .4) * s)
    im = ImageEnhance.Color(im).enhance(1 + rng.uniform(-.4, .4) * s)
    h, sat, v = im.convert('HSV').split(); h = Image.fromarray(((np.asarray(h).astype(int) + int(rng.uniform(-10, 10) * s)) % 256).astype('uint8'))
    im = Image.merge('HSV', (h, sat, v)).convert('RGB')
    a = np.asarray(im).astype('float32') / 255; a = a ** rng.uniform(1 - .35 * s, 1 + .35 * s)                  # gamma (exposure)
    a *= np.array([1 + rng.uniform(-.12, .12) * s, 1, 1 + rng.uniform(-.12, .12) * s])                           # colour temperature (warmer / cooler)
    im = Image.fromarray((np.clip(a, 0, 1) * 255).astype('uint8'))
    if rng.random() < .5 * s: im = im.filter(ImageFilter.GaussianBlur(rng.uniform(.3, 1.3)))                     # soft focus
    return np.asarray(im)

def apply_recipe(img, rng, recipe):
    """recipe = ingredients joined by '+', applied in order: flip | rrcS (min crop area S) | photoS (jitter strength S)."""
    for op in recipe.split('+'):
        if op == 'flip': img = _flip_rot90(img, rng)
        elif op.startswith('rrc'): img = random_resized_crop(img, rng, min_area=float(op[3:]))
        elif op.startswith('photo'): img = photometric(img, rng, float(op[5:]))
        elif op != 'none': raise ValueError(f'unknown ingredient {op!r} (this notebook knows flip, rrcS, photoS)')
    return img

See it work: each row is one training photo; each column adds one more ingredient.

In [ ]:
steps_ = ['none', 'flip', 'flip+rrc0.1', 'flip+rrc0.1+photo1', 'flip+rrc0.1+photo1', 'flip+rrc0.1+photo1']
demo = np.random.RandomState(3).choice(TRAIN, 3, replace=False)
show([apply_recipe(pv_img[i], np.random.default_rng(100 * c + int(i)), r) for i in demo for c, r in enumerate(steps_)],
     [(r if c < 4 else f'full recipe, draw {c - 2}') for _ in demo for c, r in enumerate(steps_)], ncols=len(steps_), cell=2.2)

> 🔧 **Try it.** Change `RECIPE` in the knobs cell (e.g. `'flip+rrc0.5+photo1'` for milder zoom, or `'flip+photo1'` for no zoom) and re-run from step 8. Which looks more like the coffee photos in step 2?

The augmentation runs on the **CPU**, in the data-loader workers, while the GPU trains. If the CPUs cannot keep up, the GPU sits idle. A quick check of how many augmented photos per second your CPUs can make:

In [ ]:
t0 = time.time(); [apply_recipe(pv_img[i], np.random.default_rng(i), RECIPE) for i in TRAIN[:200]]; per_worker = 200 / (time.time() - t0)
print(f'one CPU worker makes ~{per_worker:,.0f} augmented photos/s -> {NUM_WORKERS} workers ~{per_worker * NUM_WORKERS:,.0f}/s (the GPU consumes ~{GPU_IMG_S:,.0f}/s)')
print('data pipeline can keep the GPU busy ✅' if per_worker * NUM_WORKERS >= GPU_IMG_S * 0.5 else 'CPU-bound ⚠️ the GPU will wait for data: more workers or a bigger CPU request would speed this up (an infrastructure lesson!)')

Now re-train with the recipe: **same model, same settings, same seed.** The only change is the data. Each training photo is shown twice per epoch, once clean and once freshly augmented (a new random draw every epoch),
so the augmented run does twice as many photos and takes roughly twice as long.

In [ ]:
augmented, hist_aug, info_aug = fit(augment=lambda img, rng: apply_recipe(img, rng, RECIPE))
print(f'\naugmented model trained in {info_aug["train_seconds"]:.0f}s | peak GPU memory {info_aug["peak_gpu_gb"]:.1f} GiB')

### Compare: same field test, two models

In [ ]:
res_aug, score_aug = evaluate(augmented)
probe_aug = probe(augmented)
cmp_ = pd.DataFrame({'baseline': res_base, 'augmented': res_aug}); cmp_['change'] = cmp_['augmented'] - cmp_['baseline']
cmp_.loc['backdrop-only accuracy (lower = relies less on the backdrop)'] = [probe_base['backdrop only (no leaf!)'], probe_aug['backdrop only (no leaf!)'], probe_aug['backdrop only (no leaf!)'] - probe_base['backdrop only (no leaf!)']]
cmp_.round(3)

In [ ]:
score_hist({'baseline': score_base, 'augmented': score_aug}, cuts={'baseline': res_base[CUT_KEY], 'augmented': res_aug[CUT_KEY]})

Is the change real, or luck of which plants landed in the locked half? RoCoLe has four photos per plant, so we **bootstrap over plants**: re-draw the plants with replacement many times and recompute the AUC gain each time.
The spread of those gains is a 95% confidence interval. If it excludes zero, the improvement is unlikely to be chance *for these plants*.

In [ ]:
def bootstrap_gain(s_new, s_old, mask, n=1000, seed=0):
    rng = np.random.RandomState(seed); lp = np.unique(plants[mask]); by = {p: np.where((plants == p) & mask)[0] for p in lp}; d = []
    for _ in range(n):
        idx = np.concatenate([by[p] for p in rng.choice(lp, len(lp))]); d.append(roc_auc_score(ro_sick[idx], s_new[idx]) - roc_auc_score(ro_sick[idx], s_old[idx]))
    return float(np.mean(d)), np.percentile(d, [2.5, 97.5])
for name, mask in (('dev', DEV), ('LOCKED', LOCKED)):
    g, (lo, hi) = bootstrap_gain(score_aug, score_base, mask)
    print(f'field AUC gain from augmentation, {name:6s} half: {g:+.3f}   95% CI [{lo:+.3f}, {hi:+.3f}]  ' + ('-> clearly above zero' if lo > 0 else '-> cannot rule out zero' if hi >= 0 else '-> clearly negative'))
error_gallery(score_aug)

### Reading it honestly

* **What the bootstrap does and does not tell you.** It re-draws *plants*, so its interval says how much the answer depends on *which plants happened to be in the test half*.
  It says **nothing** about how much the answer depends on *the random seed of this training run*, and that second source of noise is often the larger one.
* **Look at several things together**, not one: did lab accuracy change, did the field AUC change, and did the share called sick and the accuracy at the tuned cut-off move?
* **One seed, one run.** In the instructors' own runs the *baseline alone* moved by about 0.1 AUC from one seed to the next. A single before/after pair is a hint, not a verdict. Step 9 shows the seed-to-seed spread and lets you add your own runs.
* The recipe was **chosen on the dev half**, so the `LOCKED` half is the fairer judge of it.
* This is still **one field, one phone, one annotator.** Nothing here says the model is ready for farmers.

> ✅ **Checkpoint 4.** You have two trained models, a comparison table, and a bootstrap interval, and you know what that interval does *not* cover.

---
## Step 9 · Scorecard, cost, and saving your work

In [ ]:
cost = pd.DataFrame({n: {'epochs': i['epochs'], 'photos per epoch': i['photos_per_epoch'], 'training time (s)': round(i['train_seconds']),
                         'photos/s (mean)': round(h['img_per_s'].mean()), 'peak GPU memory (GiB)': round(i['peak_gpu_gb'], 1)}
                     for n, (h, i) in {'baseline': (hist_base, info_base), 'augmented': (hist_aug, info_aug)}.items()})
print('What it cost:'); print(cost.to_string())
fig, ax = plt.subplots(1, 2, figsize=(10, 3.2))
for n, h in (('baseline', hist_base), ('augmented', hist_aug)): ax[0].plot(h.epoch, h.train_loss, marker='o', label=n); ax[1].plot(h.epoch, h.val_acc, marker='o', label=n)
ax[0].set_title('training loss'); ax[1].set_title('validation accuracy (lab photos)')
for a in ax: a.set_xlabel('epoch'); a.set_xticks(hist_base.epoch); a.legend(frameon=False)
plt.tight_layout(); plt.show()

Save both models and the numbers. The **default location is your home folder**, which lives on your workspace volume, so the files survive restarting the notebook server (ask your instructor if you are unsure where yours points).
In the next exercises you will run training as *batch jobs* that cannot see this notebook's memory, which is why the model must be written to disk with everything needed to reload it.

In [ ]:
OUT = Path(os.environ.get('CROPGUARD_OUT', Path.home() / 'cropguard-artifacts')) / 'exercise1'; OUT.mkdir(parents=True, exist_ok=True)
for name, model, res in (('baseline', baseline, res_base), ('augmented', augmented, res_aug)):
    torch.save({'state_dict': model.state_dict(), 'arch': 'convnext_tiny', 'classes': CLASSES, 'healthy_class_ids': HEALTHY.tolist(),
                'recipe': None if name == 'baseline' else RECIPE, 'metrics': res}, OUT / f'{name}.pt')
(OUT / 'results.json').write_text(json.dumps({'knobs': dict(SEED=SEED, EPOCHS=EPOCHS, BATCH_SIZE=BATCH_SIZE, LR=LR, WEIGHT_DECAY=WEIGHT_DECAY, LABEL_SMOOTHING=LABEL_SMOOTHING, RECIPE=RECIPE),
                                              'baseline': res_base, 'augmented': res_aug, 'cost': cost.to_dict()}, indent=1, default=float))
print({p.name: f'{p.stat().st_size / 1e6:.0f} MB' for p in OUT.iterdir()}, '->', OUT)

# A running log of every full run you make, so you can compare seeds (QUICK smoke-test runs are not logged).
AUC = 'field LOCKED: sick-vs-healthy AUC (ranking)'
if not QUICK:
    with open(OUT / 'runs.jsonl', 'a') as f: f.write(json.dumps({'seed': SEED, 'recipe': RECIPE, 'epochs': EPOCHS, 'baseline': res_base, 'augmented': res_aug}, default=float) + '\n')

### How noisy is one run? The instructors' seed table

Training is random: the shuffling, the starting weights of the new classifier layer and (for the augmented model) every augmentation draw all depend on the seed.
The instructors repeated this *entire* notebook with six seeds (on an NVIDIA GB10 GPU, so your numbers will differ somewhat). Field AUC on the **locked** half, the same metric as the table above:

| seed | baseline | + augmentation | difference |
|---|---|---|---|
| 42 (the default) | 0.862 | 0.870 | +0.008 |
| 7 | 0.917 | 0.918 | +0.001 |
| 123 | 0.811 | 0.890 | +0.079 |
| 1 | 0.754 | 0.803 | +0.049 |
| 2 | 0.849 | 0.898 | +0.049 |
| 3 | 0.895 | 0.901 | +0.006 |
| **mean ± sd** | **0.848 ± 0.059** | **0.880 ± 0.041** | **+0.032 ± 0.032** |

Read it like this:

* **One run of the *baseline* alone ranges from 0.75 to 0.92.** That spread is larger than the effect we are trying to measure, so a single before/after pair cannot settle the question. It is also noise the bootstrap in step 8 cannot see.
* **Averaged over six seeds the recipe helps a little:** about +0.03 AUC on the locked half (higher in all six seeds) and about +0.04 on the dev half (higher in four of six). That is modest next to the +0.2 we measured on a *frozen* ResNet-18,
  which fits our research finding that the recipe's benefit depends on the backbone and is largest for weak features.
* The augmented runs also varied a bit less from seed to seed (sd 0.04 against 0.06). With six seeds that is a hint, not a result.
* **Lab accuracy did not change** (97.3% vs 97.2%), and the default 0.5 cut-off stayed useless for both (balanced accuracy 0.52 vs 0.53, with about 96–97% of leaves called sick).
  A cut-off tuned on the dev half lifts balanced accuracy on the locked half to about 0.78 (baseline) and 0.81 (augmented).

> 🔧 **Try it.** Change `SEED` in the knobs cell to 7 (or any number) and re-run from there (about 4 minutes on a fast GPU). The cell below keeps a log of every full run you make.
> In class, each pair can use a different seed and we pool the "difference" column: ten runs say more than one.

In [ ]:
runs = [json.loads(l) for l in open(OUT / 'runs.jsonl')] if (OUT / 'runs.jsonl').exists() else []
if runs:
    ledger = pd.DataFrame({'seed': [r['seed'] for r in runs], 'baseline AUC': [r['baseline'][AUC] for r in runs], 'augmented AUC': [r['augmented'][AUC] for r in runs]}).drop_duplicates('seed', keep='last').set_index('seed')
    ledger['difference'] = ledger['augmented AUC'] - ledger['baseline AUC']
    print(f'{len(ledger)} full run(s) logged in {OUT / "runs.jsonl"}'); display(ledger.round(3))
    if len(ledger) >= 3: print(f'mean difference {ledger.difference.mean():+.3f}, sd {ledger.difference.std():.3f}')
else: print('no full runs logged yet (QUICK runs are not logged)')

---
## Reflection (discuss with your neighbour, ~5 minutes)

1. **Your workspace.** You had a GPU, a data volume and a ready Python environment within minutes, and it all lives in a pod you can throw away.
   What would it have cost you to set up the same thing by hand (drivers, CUDA, PyTorch builds, copying 1.5 GB of data)? What do you lose if the pod is deleted, and what survives?
2. **The test decides the story.** Lab accuracy was excellent while field ability was poor. What *kind* of data would you want in order to trust this model with a farmer? Who would collect it?
3. **The manual loop.** You changed one thing (the augmentation recipe) and waited a few minutes for one answer. There are at least seven more knobs in the first cell
   (epochs, learning rate, weight decay, label smoothing, batch size, the recipe's crop and jitter strengths). How long would a careful search over them take by hand? Could you run them in parallel from this notebook?
4. **Honesty.** After one run, how sure were you about the sign of the augmentation effect? The seed table shows what a *single* run can and cannot tell you.
   How many runs would you want before telling a colleague whether the recipe is worth keeping, and how would you run them?
5. **Ranking is not deciding.** The model ranked field photos reasonably but its default decision was useless, and the cut-off we tuned only fits one plantation.
   Who should choose the cut-off: the data scientist, the agronomist, the app team? What does a wrong answer cost a farmer in each direction (a missed disease versus needless spraying)?

## What's next

| Exercise | Kubeflow piece | Question it answers |
|---|---|---|
| **2** | **Katib** | "Which settings are best?" Let the cluster try many in parallel, 1 GPU each, scheduled by Kueue, judged on photos we are allowed to choose with (`dev`), never `locked`. |
| **3** | **Kubeflow Trainer** | "Can we train the winner faster?" A 2-GPU distributed job that writes the model back to a volume. |
| **4** | **Kubeflow Pipelines** | "Can we do it repeatably?" Wrap prepare → train → evaluate → gate into a pipeline. |
| **5** | **KServe** | "How does the farmer's phone get an answer?" Serve the saved model behind an HTTP endpoint. |

The code you read in steps 4 and 8 (`make_model`, `LeafDataset`, `fit`, `apply_recipe`, `evaluate`) is the kind of code the later exercises reuse as batch scripts and pipeline components.

## If something goes wrong

| Symptom | Likely cause and fix |
|---|---|
| `CUDA_OK` is `False` | The notebook server has no GPU. Ask an instructor; do not continue (the notebook falls back to a tiny CPU smoke test only). |
| `CUDA out of memory` | Lower `BATCH_SIZE` to 32 or 16 and re-run from the knobs cell. Another notebook of yours may be holding the GPU: *Kernel → Shut Down All Kernels*, then retry. |
| `FileNotFoundError: Could not find the CropGuard data volume` | The PVC is not mounted, or is mounted somewhere unusual. Run `!df -h`, find the folder with `MANIFEST.json`, and set `CROPGUARD_DATA`. |
| `DataLoader worker ... exited unexpectedly` / `bus error` | Not enough shared memory or RAM for the workers. Set `NUM_WORKERS = 2` (or `0`) and re-run from the knobs cell. |
| Training is much slower than the projection | The CPUs are the bottleneck (see the data-pipeline check in step 8) or another job shares the GPU: check `nvidia-smi` in a terminal. |
| Kernel dies when loading the data | The memory limit is too low for ~1.5 GB of arrays. Tell an instructor. |
| Want to start over | *Kernel → Restart Kernel and Run All Cells*. This notebook only reads the data volume, so re-running cannot damage it. |